# Data science and clear research results

Second teaching day, 11 November 2026. Start JupyterLab in this folder. All inputs are synthetic. Attempt each exercise before reading its worked answer. See README.md for setup.

## Household saving and income

How does saving vary with income, household size, and region?

Our outputs will be a checked dataset, two figures, and a descriptive table.
All records are synthetic. Income, consumption, and saving are EUR per month.

## The observation and its units

One household is one observation. A negative saving amount has an economic meaning.

In [ ]:
income = 1000
consumption = 1100
saving = income - consumption
print(saving)

```text
-100
```

## A figure and its reader

Show the same observations with vague labels and with an informative presentation.

What do the axes measure? Where is zero saving? Which households enter the figure?

Keep these questions in mind while we prepare the data.

## The research workflow

Question → inspect → prepare → summarise → plot → explain → reproduce

At each stage, make a check that addresses the calculation.
A plausible picture can conceal duplicated records or an incorrect denominator.

## Opening the lesson

Open codes/day-2/data_science.ipynb in the research-methods environment.
Start JupyterLab from codes/day-2/ and keep the data/ folder beside the notebook.

Before class, add plotnine in a terminal:

conda install -n research-methods -c conda-forge "plotnine>=0.15,<0.16"

## Imports and the working folder

Run the notebook from this lesson's folder.

In [ ]:
from pathlib import Path
import pandas as pd
import plotnine as pn
from IPython.display import HTML, display
from plotnine import (
    ggplot, aes, geom_point, geom_hline, geom_histogram,
    geom_boxplot, geom_col, facet_wrap, labs,
    theme_minimal, theme_gray, theme, element_blank, coord_flip,
)
print(Path.cwd().name)

```text
day-2
```

## A CSV becomes a DataFrame

A DataFrame is a labelled table. Shape reports rows and columns.

In [ ]:
raw = pd.read_csv("data/raw/households.csv")
print(raw.shape)
print(raw.columns.tolist())

```text
(201, 5)
['household_id', 'income', 'consumption', 'size', 'region_code']
```

## Three observations

Identify the household ID, monetary columns, number of persons, and region code.

In [ ]:
print(raw.head(3).to_string(index=False))

```text
household_id income  consumption  size region_code
        H001   1510       1610.0     1           N
        H002   1996       1620.0     2           S
        H003   4014       3200.0     3           E
```

## Types and missing entries

An invalid text token can prevent a money column from having a numeric type.

In [ ]:
print(raw.dtypes.to_string())
print(raw.isna().sum().to_string())

```text
household_id        str
income              str
consumption     float64
size              int64
region_code         str
household_id    0
income          1
consumption     3
size            0
region_code     0
```

## Column selection

A column name selects a Series. A list of names selects a DataFrame.

In [ ]:
print(raw["size"].head(3).to_list())
money = raw[["income", "consumption"]]
print(money.shape)

```text
[1, 2, 3]
(201, 2)
```

## Row selection with loc

loc[row condition, columns] selects the desired observations and fields.

In [ ]:
large = raw.loc[raw["size"] >= 4, ["household_id", "size"]]
print(large.head(3).to_string(index=False))
print(len(large))

```text
household_id  size
        H004     4
        H005     5
        H009     4
80
```

## Exercise 1: a subset

Predict whether the result can contain a household of size one. Attempt before advancing.

In [ ]:
# Select size == 2 and region_code == "N".
# Keep household_id, size, and region_code.
# Print three rows and the number of selected observations.

## Worked example: two conditions

Each comparison operates across rows. Parentheses separate the conditions.

In [ ]:
selected = raw.loc[
    (raw["size"] == 2) & (raw["region_code"] == "N"),
    ["household_id", "size", "region_code"],
]
print(selected.head(3).to_string(index=False))
print(len(selected))

```text
household_id  size region_code
        H017     2           N
        H037     2           N
        H057     2           N
10
```

## An early plot from complete rows

This preview still contains the duplicate. We will define the analysis sample explicitly.

In [ ]:
preview = raw.copy()
preview["income"] = pd.to_numeric(preview["income"], errors="coerce")
preview = preview.dropna(subset=["income", "consumption"]).copy()
preview["saving"] = preview["income"] - preview["consumption"]
preview_plot = ggplot(preview, aes("income", "saving")) + geom_point()

## The preview figure

One point represents a row. Duplicate rows can overplot each other.

In [ ]:
preview_plot

## Preserving the raw input

Transform an independent table and keep the supplied raw file unchanged.

In [ ]:
clean = raw.copy()
print(raw.shape, clean.shape)

```text
(201, 5) (201, 5)
```

## Exact duplicates and household IDs

An exact duplicate differs from a repeated ID with conflicting measurements.

In [ ]:
print(clean.duplicated().sum())
print(clean["household_id"].duplicated().sum())
clean = clean.drop_duplicates().copy()
assert clean["household_id"].is_unique
print(len(clean))

```text
1
1
200
```

## Numeric conversion

Two income entries and three consumption entries are unusable after conversion.

In [ ]:
clean["income"] = pd.to_numeric(clean["income"], errors="coerce")
print(clean[["income", "consumption"]].isna().sum().to_string())

```text
income         2
consumption    3
```

## The exclusion record

Inspect the affected rows before deciding how to handle them.

In [ ]:
incomplete = clean.loc[
    clean[["income", "consumption"]].isna().any(axis=1),
    ["household_id", "income", "consumption"],
]
print(incomplete.to_string(index=False))

```text
household_id  income  consumption
        H008     NaN       3860.0
        H023     NaN       1910.0
        H032  2068.0          NaN
        H068  4266.0          NaN
        H102  1738.0          NaN
```

## The complete-record policy

Explain how exclusions change the sample. Zero-filling would invent money values.

In [ ]:
before = len(clean)
clean = clean.dropna(subset=["income", "consumption"]).copy()
print("Excluded:", before - len(clean), "Retained:", len(clean))
assert len(clean) == 195
assert (clean["size"] > 0).all()
assert (clean[["income", "consumption"]] >= 0).all().all()

```text
Excluded: 5 Retained: 195
```

## Vectorised calculations

Arithmetic operates element by element, matching the Series index labels.

In [ ]:
clean["saving"] = clean["income"] - clean["consumption"]
clean["income_per_person"] = clean["income"] / clean["size"]
print(clean[["income", "consumption", "saving"]].head(3).to_string(index=False))

```text
 income  consumption  saving
 1510.0       1610.0  -100.0
 1996.0       1620.0   376.0
 4014.0       3200.0   814.0
```

## Undefined shares

where retains positive denominators and makes the others missing.

In [ ]:
denominator = clean["income"].where(clean["income"] > 0)
clean["saving_share"] = clean["saving"] / denominator
print("Zero incomes:", (clean["income"] == 0).sum())
print("Defined shares:", clean["saving_share"].notna().sum())

```text
Zero incomes: 2
Defined shares: 193
```

## Exercise 2: zero-income households

Predict the selected count and distinguish missing inputs from undefined ratios.

In [ ]:
# Select zero-income households.
# Print income, consumption, saving, and saving_share.
# Explain whether to exclude them from saving-amount totals.

## Worked example: amounts and shares

Both households remain in amount summaries. Share summaries have a smaller denominator.

In [ ]:
zero_income = clean.loc[
    clean["income"] == 0,
    ["income", "consumption", "saving", "saving_share"],
]
print(zero_income.to_string(index=False))
assert zero_income["saving_share"].isna().all()

```text
 income  consumption  saving  saving_share
    0.0       2080.0 -2080.0           NaN
    0.0       1570.0 -1570.0           NaN
```

## A region lookup

Each code maps to one region name. Many households can have the same code.

In [ ]:
regions = pd.read_csv("data/raw/regions.csv")
print(regions.to_string(index=False))
assert regions["region_code"].is_unique

```text
region_code region
          N  North
          S  South
          E   East
          W   West
```

## A checked left join

Cardinality and match checks protect the household sample.

In [ ]:
n_before = len(clean)
df = clean.merge(
    regions, on="region_code", how="left",
    validate="many_to_one", indicator=True,
)
assert len(df) == n_before
assert df["_merge"].eq("both").all()
print(df["_merge"].value_counts().to_string())

```text
_merge
both          195
left_only       0
right_only      0
```

## A duplicate key multiplies records

Extra rows can silently change the weights in a statistic.

In [ ]:
bad_lookup = pd.concat([regions, regions.iloc[[0]]], ignore_index=True)
unchecked = clean.merge(bad_lookup, on="region_code", how="left")
print("Before:", len(clean), "After:", len(unchecked))

```text
Before: 195 After: 245
```

## Grouped summaries

size counts group rows. count counts non-missing values in the selected column.

In [ ]:
summary = df.groupby("region", as_index=False).agg(
    n_households=("household_id", "size"),
    mean_saving=("saving", "mean"),
    median_saving=("saving", "median"),
    n_shares=("saving_share", "count"),
    mean_share=("saving_share", "mean"),
)
print(summary.round(2).to_string(index=False))

```text
region  n_households  mean_saving  median_saving  n_shares  mean_share
  East            49       351.84          366.0        48        0.12
 North            50       223.12          171.0        49        0.10
 South            49       351.04          212.0        49        0.11
  West            47       274.91          109.0        47        0.07
```

## Mean household share and pooled share

The household mean weights defined ratios equally. The pooled measure compares total saving with total income.

In [ ]:
mean_share = df["saving_share"].mean()
pooled_share = df["saving"].sum() / df["income"].sum()
print(f"Household mean: {mean_share:.2%}")
print(f"Pooled share: {pooled_share:.2%}")
print("Households:", len(df), "Defined shares:", df["saving_share"].count())

```text
Household mean: 9.91%
Pooled share: 9.60%
Households: 195 Defined shares: 193
```

## Exercise 3: a regional shortfall rate

A Boolean mean is the fraction of True observations. Explain its denominator.

In [ ]:
# Create negative_saving = saving < 0.
# For each region, calculate count and the fraction with negative saving.
# Include the zero-income households.

## Worked example: a Boolean group mean

A value of 0.20 would mean 20% of that region's retained households.

In [ ]:
df["negative_saving"] = df["saving"] < 0
shortfalls = df.groupby("region", as_index=False).agg(
    households=("household_id", "size"),
    shortfall_rate=("negative_saving", "mean"),
)
print(shortfalls.round(3).to_string(index=False))

```text
region  households  shortfall_rate
  East          49           0.245
 North          50           0.300
 South          49           0.245
  West          47           0.383
```

## The plot components

- Data supplies observations.
- Aesthetic mappings connect columns to position, colour, or size.
- Geometric objects draw the marks.
- Statistical transformations calculate counts or summaries.
- Scales, facets, and coordinates organise comparisons.
- A theme controls surrounding presentation.

## Data and position mappings

The point layer inherits the data and mappings. The + operator combines plotting components.

In [ ]:
base = ggplot(df, aes(x="income", y="saving"))
points = base + geom_point()

## The point layer

Which units and reference values would help the reader?

In [ ]:
points

## A constant colour

A setting outside aes applies the same colour to every point.

In [ ]:
constant = base + geom_point(color="#245b78", alpha=0.6)

## Colour mapped to region

A mapping varies the graphical attribute with the variable and supplies a guide.

In [ ]:
coloured = (
    ggplot(df, aes("income", "saving", color="region"))
    + geom_point(alpha=0.6)
    + labs(color="Region")
)

## The region colour mapping

The legend explains the mapping. Overlap may make group comparisons difficult.

In [ ]:
coloured

## Facets and shared scales

Facets repeat a design across groups. Fixed scales support comparisons.

In [ ]:
faceted = (
    base + geom_point(color="#245b78", alpha=0.6)
    + facet_wrap("region", ncol=2)
    + theme_minimal(base_size=16)
)

## The regional panels

Every retained household appears in exactly one panel.

In [ ]:
faceted

## A distribution plot

The histogram calculates counts within bins. The bin width is an analytical choice.

In [ ]:
histogram = (
    ggplot(df, aes("saving"))
    + geom_histogram(binwidth=250, boundary=0, fill="#245b78")
    + labs(x="Monthly saving (EUR)", y="Households")
    + theme_minimal(base_size=16)
)

## The saving distribution

All retained households have defined saving amounts, including negative amounts.

In [ ]:
histogram

## A supplied summary height

geom_col uses supplied heights. geom_bar normally counts observations.

In [ ]:
regional_bars = (
    ggplot(summary, aes("region", "mean_saving"))
    + geom_col(fill="#245b78")
    + labs(x="Region", y="Mean monthly saving (EUR)")
    + theme_minimal(base_size=16)
)

## Regional means

The bars compare mean saving amounts. Their baseline represents zero.

In [ ]:
regional_bars

## Within-region variation

Boxplots summarise distributions. Their centres indicate medians.

In [ ]:
boxes = (
    ggplot(df, aes("region", "saving"))
    + geom_boxplot(fill="#b9d3df", outlier_alpha=0.4)
    + labs(x="Region", y="Monthly saving (EUR)")
    + theme_minimal(base_size=16)
)

## Regional distributions

Compare medians and spread. The whiskers do not show confidence intervals for a mean.

In [ ]:
boxes

## Exercise 4: faceted histograms

Predict how doubling the bin width would change the picture.

In [ ]:
# Plot saving with 250 EUR bins.
# Make one panel per region with common axes.
# Label the money axis and household counts.

## Worked example: conditional distributions

Common bins and scales make the panels comparable.

In [ ]:
regional_hist = (
    ggplot(df, aes("saving"))
    + geom_histogram(binwidth=250, boundary=0, fill="#245b78")
    + facet_wrap("region", ncol=2)
    + labs(x="Monthly saving (EUR)", y="Households")
    + theme_minimal(base_size=16)
)

## The faceted distributions

Each panel describes the distribution within one region.

In [ ]:
regional_hist

## A weak presentation

The observations are correct, but the reader receives little context.

In [ ]:
bad = (
    base + geom_point(color="#ef5350", size=5)
    + labs(title="Results", x="x", y="y")
    + theme_gray(base_size=12)
)

## A chart for redesign

Identify three concrete changes that would help a reader.

In [ ]:
bad

## Labels, reference line, and source

A reference line marks zero saving. A source note declares what the reader sees.

In [ ]:
good = (
    base + geom_hline(yintercept=0, color="#777777", size=0.5)
    + geom_point(color="#245b78", alpha=0.6, size=1.8)
    + labs(title="Household saving and income",
           x="Monthly income (EUR)", y="Monthly saving (EUR)",
           caption=f"Synthetic classroom data; n = {len(df)} households")
    + theme_minimal(base_size=16)
)

## The revised presentation

The observations and calculations are unchanged.

In [ ]:
good

## Scale decisions

A positive-only saving axis would hide meaningful observations.

In [ ]:
print("Minimum saving:", df["saving"].min())
print("Negative-saving households:", (df["saving"] < 0).sum())

```text
Minimum saving: -2080.0
Negative-saving households: 57
```

## An explicit category order

Sorting rows alone does not guarantee a plotting library's discrete-axis order.

In [ ]:
region_order = summary.sort_values("mean_saving")["region"].tolist()
ordered_summary = summary.copy()
ordered_summary["region"] = pd.Categorical(
    ordered_summary["region"], categories=region_order, ordered=True
)

## An ordered horizontal chart

Horizontal category labels support comparison.

In [ ]:
ordered_plot = (
    ggplot(ordered_summary, aes("region", "mean_saving"))
    + geom_col(fill="#245b78") + coord_flip()
    + labs(x="Region", y="Mean monthly saving (EUR)")
    + theme_minimal(base_size=16)
)

## The ordered comparison

State the statistic and sample when presenting the chart.

In [ ]:
ordered_plot

## Exercise 5: design for projection

Use theme(panel_grid_minor=element_blank()). Explain the reading benefit of each choice.

In [ ]:
# Start from good and facet by region.
# Keep common scales and increase text size.
# Remove minor gridlines.

## Worked example: a presentation figure

Inspect the result at the intended display size.

In [ ]:
slide_plot = (
    good + facet_wrap("region", ncol=2)
    + theme(text=pn.element_text(size=18),
            panel_grid_minor=element_blank())
)

## The presentation figure

The data, units, zero reference, and source note remain visible.

In [ ]:
slide_plot

## A table for the reader

Give the statistic, units, and sample.

Report household counts beside counts of defined shares.
Use consistent precision and explain exclusions.
Keep full precision for calculations and round the display.

## The numerical summary

The two counts identify the denominators used by the money and share summaries.

In [ ]:
table = summary.copy()
print(table.round(2).to_string(index=False))

```text
region  n_households  mean_saving  median_saving  n_shares  mean_share
  East            49       351.84          366.0        48        0.12
 North            50       223.12          171.0        49        0.10
 South            49       351.04          212.0        49        0.11
  West            47       274.91          109.0        47        0.07
```

## Readable percentage labels

Convert a fractional share to a percentage display exactly once.

In [ ]:
display_table = table.copy()
display_table["Mean share (%)"] = 100 * display_table["mean_share"]
display_table = display_table.rename(columns={
    "region": "Region", "n_households": "Households",
    "n_shares": "Defined shares", "mean_saving": "Mean saving (EUR)",
})

## A compact display table

Note: synthetic, unweighted 2026 households. Five incomplete money records excluded after removing one exact duplicate.

In [ ]:
columns = ["Region", "Households", "Defined shares",
           "Mean saving (EUR)", "Mean share (%)"]
html_table = display_table[columns].to_html(
    index=False, float_format=lambda value: f"{value:.1f}"
)
display(HTML(html_table))

## Exercise 6: median income per person

Label the new result as EUR per person per month.

In [ ]:
# Calculate median income per person by region.
# Join this four-row result to table using region.
# Verify four rows and no unmatched regions.

## Worked example: combining summaries

The key checks prevent accidental multiplication of summary rows.

In [ ]:
medians = df.groupby("region", as_index=False).agg(
    median_income_per_person=("income_per_person", "median")
)
extended = table.merge(
    medians, on="region", how="left", validate="one_to_one"
)
assert len(extended) == 4
assert extended["median_income_per_person"].notna().all()
median_columns = ["region", "median_income_per_person"]
print(extended[median_columns].round(1).to_string(index=False))

```text
region  median_income_per_person
  East                    1151.0
 North                    1007.5
 South                     998.0
  West                    1131.0
```

## Export dimensions and formats

PNG suits screens. PDF preserves vector marks for print. Inspect exports at their intended size.

In [ ]:
output = Path("outputs")
output.mkdir(exist_ok=True)
good.save(output / "saving_income.png", width=8, height=4.5,
          dpi=200, verbose=False)
good.save(output / "saving_income.pdf", width=8, height=4.5,
          verbose=False)

## Unrounded numerical exports

index=False avoids an unintended row-index column.

In [ ]:
summary.to_csv(output / "regional_summary.csv", index=False)
clean.to_csv(output / "checked_households.csv", index=False)
(output / "regional_table.html").write_text(html_table)
print((output / "regional_summary.csv").is_file())

```text
True
```

## Exercise 7: an independent report

Answer how saving differs across regions.

1. Rebuild the checked sample from the raw CSVs.
2. Produce one labelled table and two figures with different purposes.
3. Include units, sample counts, and the synthetic-data note.
4. Add four meaningful checks and explain the design choices.
5. Save the outputs and restart the notebook.

This is classroom practice, separate from graded take-home work.

## Worked workflow: preparing the sample

This cell starts from files rather than remembered notebook values.

In [ ]:
fresh = pd.read_csv("data/raw/households.csv").drop_duplicates()
fresh["income"] = pd.to_numeric(fresh["income"], errors="coerce")
fresh = fresh.dropna(subset=["income", "consumption"]).copy()
assert fresh["household_id"].is_unique
assert (fresh["size"] > 0).all()
fresh["saving"] = fresh["income"] - fresh["consumption"]
positive_income = fresh["income"].where(fresh["income"] > 0)
fresh["saving_share"] = fresh["saving"] / positive_income

## Worked workflow: the region join

Check the cardinality and match status before reporting results.

In [ ]:
lookup = pd.read_csv("data/raw/regions.csv")
fresh = fresh.merge(
    lookup, on="region_code", how="left",
    validate="many_to_one", indicator=True,
)
assert len(fresh) == 195
assert fresh["_merge"].eq("both").all()
assert fresh["saving_share"].count() == 193

## Worked workflow: a final table

A final report explains both its pattern and the decisions that produced it.

In [ ]:
final_summary = fresh.groupby("region", as_index=False).agg(
    households=("household_id", "size"),
    mean_saving=("saving", "mean"),
    defined_shares=("saving_share", "count"),
)
assert final_summary["households"].sum() == len(fresh)
final_summary.to_csv(output / "lab_summary.csv", index=False)

## A bounded coding-agent task

Ask an agent to improve only the labels and theme of one figure.

Keep the sample, calculations, mappings, and scales fixed.
Inspect the change and verify the sample count.
Explain one design decision you accepted and the check you performed.

## The final reproducibility check

Restart and run all cells in order.

Can another student reproduce your table and two figures from the supplied files?
Can you explain every exclusion, denominator, mapping, and design choice?
Does each title make a claim supported by the displayed result?